
# 4. Description-based simulation

In lesson 3 the events were generated by the layout. They can also come from
an existing sequence: a Pulseq file written with another tool, or the
sequence description a scanner streams during acquisition. This lesson
simulates such descriptions directly, without restating the sequence
parameters.

**Learning objectives**

- Inspect a sequence description: events, timestamps and RF definitions.
- Simulate a description with ``from_description``, and understand the role
  of the handlers.
- Simulate a Pulseq file with ``from_pulseq``.

Previous: :doc:`03-custom-signal-model`. Next: :doc:`05-custom-operator`, which
adds a custom preparation and readout.


<div class='alert alert-info'>

# Install libraries needed for Colab

The below installation commands are needed to be run only on Google Colab.
</div>
<div class="colab-button">
            <a href="https://colab.research.google.com/github/pulserver/blochsim/blob/gh-pages/v0.0.13/notebooks/generated/autoexamples/01-framework/04-description-based-simulation.ipynb" target="_blank">
                <img src="https://colab.research.google.com/assets/colab-badge.svg"
                alt="Open In Colab"/>
            </a>
        </div>
        

In [ ]:
# Install libraries
!pip install blochsim matplotlib
!wget --quiet --no-clobber https://raw.githubusercontent.com/pulserver/blochsim/main/docs/figure_style.py
!wget --quiet --no-clobber https://raw.githubusercontent.com/pulserver/blochsim/main/examples/01-framework/fse.seq

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt

from figure_style import MUTED, PAGE_WIDTH, SERIES, legend_outside

warnings.filterwarnings("ignore")
# The gallery runs a script from its own directory; a shell may not.
SEQ_FILE = next(
    candidate
    for candidate in (Path("fse.seq"), Path("examples/01-framework/fse.seq"))
    if candidate.exists()
)

## Sequence description

A *sequence description* (:class:`~blochsim.SequenceDescription`) lists the
events of one repetition: RF pulses, ADC windows and waits, each with its
timestamp and parameters (flip angle, phase, RF definition). It is the input
of every simulation.

The default description of a simulator is generated by its layout from the
sequence parameters, and ``describe`` returns it. For the 60-degree FSE of
lesson 1:


In [ ]:
import torch

from blochsim.simulators import FSESimulator

fse = FSESimulator(ESP=5.0, TR=3000.0)
flip = torch.full((48,), 60.0)

description = fse.describe(flip=flip, ESP=5.0, TR=3000.0)
print(f"{len(description.events)} events, {len(description.adc_events)} readouts")

fig, ax = plt.subplots(figsize=(PAGE_WIDTH, 0.35 * PAGE_WIDTH))
description.plot(ax, upto_s=6 * 5e-3)
ax.set(ylim=(-8, 105))
legend_outside(ax)
plt.show()

The first 30 ms show the excitation and the refocusing pulses centred
between readouts. The description contains no gradients: the scanner stream
does not carry them, and blochsim does not read them from a Pulseq file.

## Simulation from a description

``from_description`` constructs a simulator from a description, bypassing
the layout. Only tissue properties and simulation settings are passed:


In [ ]:
from_events = FSESimulator.from_description(description, states=48, T1=1000.0, T2=80.0)
signal = from_events.simulate()

The handlers of the class are still used: they play each event and define
the dephasing between events, which the description does not specify.
``FSESimulator`` places crushers around each refocusing pulse. The same
events played with :class:`~blochsim.simulators.MRFSimulator`, whose readout
applies one unit of dephasing after each sample, describe a different
sequence:


In [ ]:
from blochsim.simulators import MRFSimulator

as_mrf = MRFSimulator.from_description(description, states=48, T1=1000.0, T2=80.0)
wrong = as_mrf.simulate()

reference = fse.simulate(flip=flip, T1=1000.0, T2=80.0, states=48)
fig, ax = plt.subplots(figsize=(0.8 * PAGE_WIDTH, 0.4 * PAGE_WIDTH))
echoes = range(1, 49)
ax.plot(
    echoes, reference.abs().reshape(-1), color=MUTED, lw=3, label="FSESimulator, layout"
)
ax.plot(
    echoes, signal.abs().reshape(-1), color=SERIES[0], label="FSE handlers, description"
)
ax.plot(
    echoes, wrong.abs().reshape(-1), color=SERIES[1], label="MRF handlers, description"
)
ax.set(xlabel="echo", ylabel="|signal| (a.u.)", xlim=(1, 48), ylim=(0, None))
legend_outside(ax)
plt.show()

With the FSE handlers the result matches the shipped simulator up to a small
offset: ``FSESimulator`` also applies the T1 recovery between trains, which is
not an event and is therefore absent from the description. With the MRF
handlers the CPMG condition is not met and the echo amplitudes oscillate and
decay. The class therefore specifies how the events are to be interpreted.

## Scanner stream

During acquisition, pulserver streams the description of the running
sequence with the raw data. :func:`~blochsim.read_mrd_description` reads it
from an MRD stream, and ``from_description`` simulates it as above. A
reconstruction can thus simulate the sequence that was actually played.

## Pulseq file

``from_pulseq`` reads the description from a ``.seq`` file, parsed by
pypulseq. The file below is a short FSE:


In [ ]:
from_file = FSESimulator.from_pulseq(SEQ_FILE, states=20)
file_signal = from_file.simulate(T1=1000.0, T2=torch.tensor([80.0, 110.0]))

read = from_file.describe()
echo_ms = torch.tensor([event.timestamp_us for event in read.adc_events]) * 1e-3
echo_ms = echo_ms - echo_ms[0] + (echo_ms[1] - echo_ms[0])
print(f"{len(read.adc_events)} echoes, {echo_ms[1] - echo_ms[0]:.1f} ms apart")
fig, axes = plt.subplots(2, 1, figsize=(PAGE_WIDTH, 0.65 * PAGE_WIDTH))
read.plot(axes[0])
axes[0].set(ylim=(-8, 200))
legend_outside(axes[0])
for row, name in enumerate(("white matter", "grey matter")):
    axes[1].plot(echo_ms, file_signal[row].abs(), "o-", color=SERIES[row], label=name)
axes[1].set(xlabel="time after excitation (ms)", ylabel="|signal| (a.u.)")
legend_outside(axes[1])
plt.show()

Echo spacing, echo train length, refocusing angle and RF shapes are read
from the file; only the tissue and the handler class are specified.

The number of blocks per repetition is taken from the ``TRSize`` definition
of the file. Each readout is timestamped at the sample where the trajectory
crosses the k-space centre, i.e. at the echo time.

## Specification

What this lesson did, stated the way you would ask an agent for it:

```text
With blochsim, read the sequence in fse.seq with FSESimulator.from_pulseq
and simulate it in white matter and grey matter (T1 1 s; T2 80 and
110 ms). Plot the events of one repetition and the signal at each echo
time. Separately, take the description of a 48-echo, 60-degree FSE and
simulate it with the FSE handlers and with the MRF handlers, to show that
the handlers decide what happens between the events.
```
